In [6]:
import numpy as np
import gymnasium as gym
import math
from stable_baselines3 import SAC
from simulation.controllers.two_wheel_robot.two_wheel_robot_base import TwoWheelRobot
import simulation.controllers.two_wheel_robot.helpers

In [7]:
class BaseAgent(gym.Env):

    def reset(self, *args, **kwargs): ...

    def render(self): ...

    def close(self): ...

    def step(self, action): ...

In [8]:
class TwoWheelRobotBase(TwoWheelRobot):
    def __init__(
        self,
        wheel_distance=1,
    ):
        self.goal = (0, 0)
        self.wheel_distance = wheel_distance
        # TODO: ensure wheel seperation is not zero

        self.reset()

    def reset(
        self,
        goal: tuple[float, float] = None,
        pos: tuple[float, float] = None,
    ):
        self.x = 0.0
        self.y = 0.0
        self.theta = 0.0
        self.v = 0.0
        self.omega = 0.0
        self.goal = (0, 0)

        if goal:
            self.goal = goal

        if pos:
            self.x = pos[0]
            self.y = pos[1]

    def state(self):
        return (
            self.x,
            self.y,
            self.theta,
        )

    def move_wheels(self, v_right, v_left):
        self.v = (v_right + v_left) * 0.5
        self.omega = (v_right - v_left) / self.wheel_distance
       

    def step(self, dt):
        self.x += self.v * math.cos(self.theta) * dt
        self.y += self.v * math.sin(self.theta) * dt
        self.theta += self.omega * dt

In [9]:
class TwoWheelRobotEnv(BaseAgent):

    def __init__(
        self,
        observation_space,
        action_space,
        robot: TwoWheelRobot,
        render_mode: str = None,
    ):
        self.observation_space = observation_space
        self.action_space = action_space
        self.robot = robot
        self.render_mode = render_mode

    def reset(self, *args, **kwargs):
        obs = [0, 0]
        info = {}

        return obs, info

    def render(self): ...

    def close(self): ...

    def step(self, action):
        self.render()

        obs = [0, 0]
        reward = 0
        terminated = True
        truncated = True
        info = {}

        return obs, reward, terminated, truncated, info

In [ ]:
robot = TwoWheelRobotBase()
robot.reset(
    goal=(4, 4),
    pos=(0, 0),
)
robot.move_wheels(1,-1)
robot.step(200)
robot.state()

In [ ]:
robot = TwoWheelRobotBase()
robot.reset(
    goal=(4, 4),
    pos=(0, 0),
)
obs = gym.spaces.Box(
    low=np.array([-1, -1]),
    high=np.array([1, 1]),
)
actions = gym.spaces.Box(
    low=np.array([-1, -1]),
    high=np.array([1, 1]),
)

env = TwoWheelRobotEnv(
    observation_space=obs,
    action_space=actions,
    robot=robot,
)
model = SAC("MlpPolicy", env, verbose=1)
model.learn(
    total_timesteps=5,
    log_interval=5,
)
env.close()


In [12]:
model.save("robot_nav")
model = SAC.load("robot_nav")
obs, info = env.reset()
action, _states = model.predict(obs, deterministic=True)
obs, reward, terminated, truncated, info = env.step(action)
if terminated or truncated:
    obs, info = env.reset()